<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 02: Feature pipeline for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook downloads new data and ingests it into hopsworks feature groups.

It performs the following steps:

1. 


### 📝 Imports

In [15]:
# top of notebook
from features import (
    build_features,
    add_calendar_features,
    add_train_lag_features,
    add_station_network_state_features,
    detect_trigger_time,
    add_reactive_early_dynamics,
    add_weather_rolling_features_if_present,
    add_station_delay_features,
    ensure_event_time,
    add_cause_flags,
    add_station_congestion_features,
    build_duration_baseline,
)


In [16]:
import os
import datetime
import pandas as pd
import requests
#import hopsworks_utils
import datetime as dt
from dotenv import load_dotenv
#import hopsworks
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

True

## 📡 Connect to Hopsworks Feature Store

In [17]:
#### it need to get fixed

try:
    project = hopsworks_utils.HopsworksInterface()
    print("Hopsworks login OK")
except Exception as e:
    project = None
    print("Hopsworks not configured / login failed (OK). Proceeding without it.")
    print("Reason:", repr(e))

#train_feature_df = project.get("train_stop_events_labeled")

#uncoment the below line when weather features are stored
#weather_df = project.get("weather_features") 


Hopsworks not configured / login failed (OK). Proceeding without it.
Reason: NameError("name 'hopsworks_utils' is not defined")


In [18]:
# Paths
CANONICAL_PATH = os.getenv("CANONICAL_PATH", "data/train_stop_events_labeled.parquet")



OUT_DIR = os.getenv("OUT_DIR", "data/feature_pipeline_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# Label settings (must match Part 01)
HORIZON_MIN = int(os.getenv("HORIZON_MIN", "60"))
DELAY_THRESHOLD_MIN = int(os.getenv("DELAY_THRESHOLD_MIN", "10"))

# Rolling windows for network-state features
ROLL_WINDOWS_MIN = [1440, 1440*2]   # minutes
WEATHER_ROLL_WINDOWS_H = [3, 6]  # hours (only used if weather columns exist)

# Split ratios (time-based, by unique dates)
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9


In [19]:
if not os.path.exists(CANONICAL_PATH):
    raise FileNotFoundError(
        f"Could not find canonical dataset at {CANONICAL_PATH}. "
        "Run Part 01 and make sure it saved train_stop_events_labeled.parquet."
    )

df = pd.read_parquet(CANONICAL_PATH)

#print("Loaded:", CANONICAL_PATH)
#print("Shape:", df.shape)
#display(df.head())


In [20]:
# Ensure required columns exist
required_cols = ["event_time", "station_code", "train_id", "delay_min",
                 "y_delay_within_horizon", "final_delay_min", "additional_delay_min"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in canonical dataset: {missing}")

df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")
df = df.dropna(subset=["event_time", "station_code", "train_id"]).copy()

# Ensure a train-run key exists
if "train_run_id" not in df.columns:
    df["date"] = df["event_time"].dt.date
    df["train_run_id"] = df["train_id"].astype(str) + "_" + df["date"].astype(str)

# Sort for point-in-time computations
df = df.sort_values(["event_time", "station_code", "train_run_id"]).reset_index(drop=True)

# Normalize reason_code
if "reason_code" in df.columns:
    df["reason_code"] = df["reason_code"].astype("string")
else:
    df["reason_code"] = pd.Series([pd.NA]*len(df), dtype="string")

# --- RENAME STEP ADDED HERE ---
weather_rename_map = {
    "temperature_2m": "weather_temperature_2m",
    "precipitation": "weather_precipitation",
    "rain": "weather_rain",
    "snowfall": "weather_snowfall",
    "windspeed_10m": "weather_windspeed_10m"
}
existing_rename = {k: v for k, v in weather_rename_map.items() if k in df.columns}
if existing_rename:
    print(f"Renaming weather columns: {list(existing_rename.keys())}")
    df = df.rename(columns=existing_rename)
# ------------------------------

print("After cleaning:", df.shape)

Renaming weather columns: ['temperature_2m', 'precipitation', 'rain', 'snowfall', 'windspeed_10m']
After cleaning: (69460, 32)


### Feature engineering (no leakage)

In [ ]:
print("⚡️ Engineering features using build_features()...")
df_feat = build_features(df)
print("✅ Feature engineering complete.", df_feat.shape)


⚡️ Engineering features using build_features()...


In [ ]:
print([c for c in ["Deviation", "observed_time", "estimated_time", "event_time"] if c in df_feat.columns])
df_check = df_feat[
    ["event_time", "Deviation", "observed_time"]
].dropna(subset=["Deviation", "observed_time"])

df_check["delta_minutes"] = (
    df_check["event_time"] - df_check["observed_time"]
).dt.total_seconds() / 60

df_check["delta_minutes"].describe()


['Deviation', 'observed_time', 'estimated_time', 'event_time']


count    9922.000000
mean       -7.500867
std        17.422977
min      -352.883333
25%        -7.000000
50%        -1.000000
75%         0.000000
max       101.894117
Name: delta_minutes, dtype: float64

In [ ]:
df_check2 = df_feat[
    ["event_time", "Deviation", "estimated_time"]
].dropna(subset=["Deviation", "estimated_time"])

df_check2["delta_minutes_est"] = (
    df_check2["event_time"] - df_check2["estimated_time"]
).dt.total_seconds() / 60

df_check2["delta_minutes_est"].describe()


count    3636.000000
mean      -16.887789
std        22.603321
min      -210.000000
25%       -20.000000
50%        -9.000000
75%        -4.000000
max        -1.000000
Name: delta_minutes_est, dtype: float64

In [ ]:
# Apply feature engineering
"""
print("⚡️ Engineering features using 'features.py'...")

df_feat = df.copy()

# Ensure event_time is timezone-aware
df_feat = ensure_event_time(df_feat)

# Add cause flags based on reason columns
df_feat = add_cause_flags(df_feat)

# Add station congestion features (lag + rolling counts)
df_feat = add_station_congestion_features(df_feat, windows_min=ROLL_WINDOWS_MIN)

# 1. Calendar
df_feat = add_calendar_features(df_feat)

# 2. Lag features
df_feat = add_train_lag_features(df_feat)

# 3. Network State
df_feat = add_station_network_state_features(df_feat, windows_min=ROLL_WINDOWS_MIN)

# 4. Trigger Detection
df_feat = detect_trigger_time(df_feat)

# 5. Reactive Dynamics
#df_feat = add_reactive_early_dynamics(df_feat)

# 6. Weather
df_feat = add_weather_rolling_features_if_present(df_feat, windows_h=WEATHER_ROLL_WINDOWS_H)

# 7. Station delay features
df_feat = add_station_delay_features(df_feat)

print("✅ Feature engineering complete.")
print("Feature table shape:", df_feat.shape)
df_feat.sort_values(by=["delay_min"], ascending=False, inplace=True)

print(df_feat.info(verbose=True))
display(df_feat.head())
"""

'\nprint("⚡️ Engineering features using \'features.py\'...")\n\ndf_feat = df.copy()\n\n# Ensure event_time is timezone-aware\ndf_feat = ensure_event_time(df_feat)\n\n# Add cause flags based on reason columns\ndf_feat = add_cause_flags(df_feat)\n\n# Add station congestion features (lag + rolling counts)\ndf_feat = add_station_congestion_features(df_feat, windows_min=ROLL_WINDOWS_MIN)\n\n# 1. Calendar\ndf_feat = add_calendar_features(df_feat)\n\n# 2. Lag features\ndf_feat = add_train_lag_features(df_feat)\n\n# 3. Network State\ndf_feat = add_station_network_state_features(df_feat, windows_min=ROLL_WINDOWS_MIN)\n\n# 4. Trigger Detection\ndf_feat = detect_trigger_time(df_feat)\n\n# 5. Reactive Dynamics\n#df_feat = add_reactive_early_dynamics(df_feat)\n\n# 6. Weather\ndf_feat = add_weather_rolling_features_if_present(df_feat, windows_h=WEATHER_ROLL_WINDOWS_H)\n\n# 7. Station delay features\ndf_feat = add_station_delay_features(df_feat)\n\nprint("✅ Feature engineering complete.")\nprint("Fea

In [ ]:
import os
import pandas as pd
import numpy as np
import json
import datetime as dt

# --- 0. PRE-REQUISITE: DEFINE FEATURE COLUMNS ---
# Define which columns are ID/Target/Future and should be excluded
ALL_KEYS = ["ActivityId", "train_id", "lag_y_delay", "InformationOwner", "scheduled_time", "estimated_time", "actual_time", "observed_time", "reason_code", "reason_text", "reason_desc", "OperationalTrainNumber", "station_code", "event_time", "event_date", "train_run_id"]
TARGETS = ["y_delay_within_horizon", "final_delay_min", "additional_delay_min"]

# Columns to exclude from Predictive Model (future leakage or reactive-only)
PRED_EXCLUDE = set(TARGETS + ["final_delay_min", "additional_delay_min", 
                              "trigger_time", "min_since_trigger", "delay_at_trigger", 
                              "delay_slope_since_trigger", "is_first10m_after_trigger"])

# Columns to exclude from Reactive Model (just the predictive target)
REACT_EXCLUDE = set(["y_delay_within_horizon"])

# Calculate the lists of columns dynamically from df_feat
pred_cols = [c for c in df_feat.columns if c not in ALL_KEYS and c not in PRED_EXCLUDE]
react_cols = [c for c in df_feat.columns if c not in ALL_KEYS and c not in TARGETS and c not in REACT_EXCLUDE]

print(f"Features detected: {len(pred_cols)} Predictive, {len(react_cols)} Reactive")


# --- 1. SPLIT LOGIC ---
df_feat["event_date"] = df_feat["event_time"].dt.date
dates = sorted(df_feat["event_date"].unique())
n = len(dates)

if n >= 3:
    n_val = max(1, int(np.floor(n * 0.15)))  
    n_test = max(1, int(np.floor(n * 0.15))) 
    n_train = n - n_val - n_test
else:
    n_train = n
    n_val = 0
    n_test = 0

train_dates = set(dates[:n_train])
val_dates   = set(dates[n_train:n_train+n_val])
test_dates  = set(dates[n_train+n_val:])

print(f"Refined Split: Train={len(train_dates)}d, Val={len(val_dates)}d, Test={len(test_dates)}d")



# --- 2. RE-SLICE DATAFRAMES ---
df_train = df_feat[df_feat["event_date"].isin(train_dates)].copy()
df_val   = df_feat[df_feat["event_date"].isin(val_dates)].copy()
df_test  = df_feat[df_feat["event_date"].isin(test_dates)].copy()

# Reactive Slices (Filter for delay >= threshold)
df_react_train = df_train[df_train["delay_min"] >= DELAY_THRESHOLD_MIN].copy()
df_react_val   = df_val[df_val["delay_min"] >= DELAY_THRESHOLD_MIN].copy()
df_react_test  = df_test[df_test["delay_min"] >= DELAY_THRESHOLD_MIN].copy()

# --- 3. REGENERATE X AND y ---
# Predictive Targets
X_pred_train = df_train[pred_cols]
y_pred_train = df_train["y_delay_within_horizon"]

X_pred_val   = df_val[pred_cols]
y_pred_val   = df_val["y_delay_within_horizon"]

X_pred_test  = df_test[pred_cols]
y_pred_test  = df_test["y_delay_within_horizon"]

# Reactive Targets
X_react_train = df_react_train[react_cols]
y_react_train = df_react_train["additional_delay_min"]

X_react_val   = df_react_val[react_cols]
y_react_val   = df_react_val["additional_delay_min"]

X_react_test  = df_react_test[react_cols]
y_react_test  = df_react_test["additional_delay_min"]

# --- 4. PACK AND SAVE ---
# Keys to keep in the final output for joining
SAVE_KEYS = ["train_run_id", "event_time", "station_code", "train_id"]

pred_train_path = os.path.join(OUT_DIR, "pred_train.parquet")
pred_val_path   = os.path.join(OUT_DIR, "pred_val.parquet")
pred_test_path  = os.path.join(OUT_DIR, "pred_test.parquet")

react_train_path = os.path.join(OUT_DIR, "react_train.parquet")
react_val_path   = os.path.join(OUT_DIR, "react_val.parquet")
react_test_path  = os.path.join(OUT_DIR, "react_test.parquet")
"""
def pack(df_split, X, y, task: str) -> pd.DataFrame:
    if df_split.empty:
        return pd.DataFrame()
        
    actual_keys = [k for k in SAVE_KEYS if k in df_split.columns]
    packed = df_split[actual_keys].copy().reset_index(drop=True)
    
    if isinstance(X, pd.DataFrame):
        X = X.reset_index(drop=True)
        packed = pd.concat([packed, X], axis=1)
    else:
        packed = packed.join(pd.DataFrame(X, columns=pred_cols if task=="pred" else react_cols))
        
    # Correctly name the target column
    if task == "pred":
        packed["y_delay_within_horizon"] = y.values
    elif task == "react":
        packed["additional_delay_min"] = y.values
    else:
        packed[f"y_{task}"] = y.values
    
    return packed
"""
def pack(df_split, X, y, task: str) -> pd.DataFrame:
    """
    Always returns a dataframe with the correct schema (even if df_split is empty),
    so parquet files reload with columns instead of (0,0).
    """
    if task == "pred":
        feat_cols = pred_cols
        target_col = "y_delay_within_horizon"
    elif task == "react":
        feat_cols = react_cols
        target_col = "additional_delay_min"
    else:
        raise ValueError(f"Unknown task={task}")

    # Keys to keep
    actual_keys = [k for k in SAVE_KEYS if (not df_split.empty and k in df_split.columns)]
    if df_split.empty:
        # Create empty keys frame with stable schema
        packed = pd.DataFrame({k: pd.Series(dtype="object") for k in SAVE_KEYS})
        packed = packed[[k for k in SAVE_KEYS]]  # keep order
    else:
        packed = df_split[actual_keys].copy().reset_index(drop=True)

    # Attach X with correct columns (even if empty)
    if isinstance(X, pd.DataFrame):
        X2 = X.copy()
        for c in feat_cols:
            if c not in X2.columns:
                X2[c] = pd.Series(dtype="float64")
        X2 = X2[feat_cols].reset_index(drop=True)
    else:
        X2 = pd.DataFrame(X, columns=feat_cols)

    packed = pd.concat([packed.reset_index(drop=True), X2.reset_index(drop=True)], axis=1)

    # Attach target with correct column name (even if empty)
    if y is None or len(y) == 0:
        packed[target_col] = pd.Series(dtype="float64")
    else:
        packed[target_col] = pd.Series(y.values)

    return packed

def save_parquet(df, path):
    if not df.empty:
        df.to_parquet(path, index=False)
        print(f"✅ Saved: {path} ({len(df)} rows)")
    else:
        print(f"⚠️ Empty split, saving schema only: {path}")
        df.to_parquet(path, index=False)

# Save
save_parquet(pack(df_train, X_pred_train, y_pred_train, "pred"), pred_train_path)
save_parquet(pack(df_val,   X_pred_val,   y_pred_val,   "pred"), pred_val_path)
save_parquet(pack(df_test,  X_pred_test,  y_pred_test,  "pred"), pred_test_path)

save_parquet(pack(df_react_train, X_react_train, y_react_train, "react"), react_train_path)
save_parquet(pack(df_react_val,   X_react_val,   y_react_val,   "react"), react_val_path)
save_parquet(pack(df_react_test,  X_react_test,  y_react_test,  "react"), react_test_path)

# Metadata (Includes feature names for Part 4!)
metadata = {
    "created_utc": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "splits": {
        "train_dates": [str(d) for d in sorted(train_dates)],
        "val_dates": [str(d) for d in sorted(val_dates)],
        "test_dates": [str(d) for d in sorted(test_dates)],
    },
    "pred_feature_columns": pred_cols,   # <--- Critical for Part 4
    "react_feature_columns": react_cols  # <--- Critical for Part 4
}
meta_path = os.path.join(OUT_DIR, "feature_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata saved to {meta_path}")

Features detected: 47 Predictive, 49 Reactive
Refined Split: Train=2d, Val=1d, Test=1d
✅ Saved: data/feature_pipeline_outputs/pred_train.parquet (48286 rows)
✅ Saved: data/feature_pipeline_outputs/pred_val.parquet (17896 rows)
✅ Saved: data/feature_pipeline_outputs/pred_test.parquet (9516 rows)
✅ Saved: data/feature_pipeline_outputs/react_train.parquet (5261 rows)
✅ Saved: data/feature_pipeline_outputs/react_val.parquet (580 rows)
⚠️ Empty split, saving schema only: data/feature_pipeline_outputs/react_test.parquet
✅ Metadata saved to data/feature_pipeline_outputs/feature_metadata.json


In [ ]:
print("df_test rows:", len(df_test))
print("df_test delay_min stats:")
print(df_test["delay_min"].describe(include="all"))

print("DELAY_THRESHOLD_MIN:", DELAY_THRESHOLD_MIN)
print("Rows in test >= threshold:", (df_test["delay_min"] >= DELAY_THRESHOLD_MIN).sum())
print("NaNs in test delay_min:", df_test["delay_min"].isna().sum())

print("test_dates:", sorted(test_dates)[:5], "...", sorted(test_dates)[-5:])
print("test day counts:")
print(df_test.groupby("event_date").size())
print("test day counts (>=threshold):")
print(df_test[df_test["delay_min"] >= DELAY_THRESHOLD_MIN].groupby("event_date").size())


df_test rows: 9516
df_test delay_min stats:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: delay_min, dtype: float64
DELAY_THRESHOLD_MIN: 10
Rows in test >= threshold: 0
NaNs in test delay_min: 9516
test_dates: [datetime.date(2026, 1, 11)] ... [datetime.date(2026, 1, 11)]
test day counts:
event_date
2026-01-11    9516
dtype: int64
test day counts (>=threshold):
Series([], dtype: int64)


In [ ]:
#List columns in training and test data
#print("Predictive feature columns:", pred_cols)

display(df_train[pred_cols].head())
#print(df_train.columns.tolist())

,ActivityType,delay_min,is_canceled,Deleted,Deviation,FromLocation,ToLocation,TrackAtLocation,hour,dow,...,weather_precipitation_rollmean_3h,weather_rain_rollmean_3h,weather_snowfall_rollmean_3h,weather_windspeed_10m_rollmean_3h,weather_temperature_2m_rollmean_6h,weather_precipitation_rollmean_6h,weather_rain_rollmean_6h,weather_snowfall_rollmean_6h,weather_windspeed_10m_rollmean_6h,station_avg_delay
0,Ankomst,2.0,False,False,None,Cst,U,1,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.479954
1,Avgang,4.0,False,False,None,Cst,"Kn,U",1,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.479954
2,Avgang,NaN,True,False,Inställt,U,Sci,x,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.479954
3,Ankomst,NaN,True,False,Inställt,U,Sci,x,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.479954
4,Ankomst,-1.0,False,False,None,U,Söc,2,4,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.479954


In [ ]:
# Verify weather columns are in the list
weather_cols_saved = [c for c in pred_cols if "weather" in c]
print(f"n📊 Verification: {len(weather_cols_saved)} weather features included in predictive model.")
if len(weather_cols_saved) > 0:
    print("Example features:", weather_cols_saved[:3])
else:
    print("⚠️ WARNING: No weather features detected in the final list!")


n📊 Verification: 17 weather features included in predictive model.
Example features: ['weather_temperature_2m', 'weather_precipitation', 'weather_rain']


In [ ]:
print("done")

### ✂️ Time-based train/val/test split (no leakage)

### 🧩 Build predictive vs reactive feature matrices

### 💾 Save outputs + feature metadata